In [2]:
!pip install numpy


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.5 MB 1.3 MB/s eta 0:00:10
   --- ------------------------------------ 1.0/12.5 MB 2.1 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/12.5 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.5 MB 2.3 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.5 MB 2.2 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.5 MB 2.2 MB/s eta 0:00:05
   ---------- ----------------------------- 3.4/12.5 MB 2.1 MB/s eta 0:00:05
   ------------ --------------------------- 3.9/12.5 MB 2.2 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.5 MB 2.1 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/12.5 MB 2.2 MB/s eta 0:00:04
   ---------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import math
import os
os.chdir(r"C:\Users\Mineth De Croos\Desktop\AI\Arch labs\attention_hardware")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Mineth De Croos\Desktop\AI\Arch labs\attention_hardware


In [4]:
# Input embeddings
X = np.array([
    [1, 0, 1, 0],   # "cat"
    [0, 1, 0, 1],   # "sat"
    [1, 1, 0, 0]    # "mat"
], dtype=float)

# Weight matrices
WQ = np.array([
    [ 1,  0,  1,  0],
    [ 0,  1,  0,  1],
    [ 1,  0, -1,  0],
    [ 0,  1,  0, -1]
], dtype=float)

WK = np.array([
    [ 0,  1,  0,  1],
    [ 1,  0,  1,  0],
    [ 0, -1,  0,  1],
    [ 1,  0, -1,  0]
], dtype=float)

WV = np.array([
    [ 1,  0,  0,  1],
    [ 0,  1,  1,  0],
    [ 1,  0,  0, -1],
    [ 0,  1, -1,  0]
], dtype=float)

print("X =")
print(X)

X =
[[1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 1. 0. 0.]]


In [ ]:
# Step 1: Compute Q, K, V
Q = X @ WQ
K = X @ WK
V = X @ WV

# @ is the matrix multiplication operator in Python (equivalent to np.dot)

print("Q =")
print(Q)
print("\nK =")
print(K)
print("\nV =")
print(V)

Q =
[[2. 0. 0. 0.]
 [0. 2. 0. 0.]
 [1. 1. 1. 1.]]

K =
[[0. 0. 0. 2.]
 [2. 0. 0. 0.]
 [1. 1. 1. 1.]]

V =
[[2. 0. 0. 0.]
 [0. 2. 0. 0.]
 [1. 1. 1. 1.]]


In [6]:
# Step 2: Compute scores
d_k = Q.shape[1]  # embedding size = 4
scores = Q @ K.T

print("Scores =")
print(scores)

Scores =
[[0. 4. 2.]
 [0. 0. 2.]
 [2. 2. 4.]]


In [9]:
scaled_scores = scores / math.sqrt(d_k)

print("Scaled scores =")
print(scaled_scores)

Scaled scores =
[[0. 2. 1.]
 [0. 0. 1.]
 [1. 1. 2.]]


In [10]:
# Step 4: Softmax
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

attention_weights = softmax(scaled_scores)

print("Attention weights =")
print(np.round(attention_weights, 2))

Attention weights =
[[0.09 0.67 0.24]
 [0.21 0.21 0.58]
 [0.21 0.21 0.58]]


In [11]:
# Step 5: Output = attention_weights × V
output = attention_weights @ V

print("Output =")
print(np.round(output, 2))

Output =
[[0.42 1.58 0.24 0.24]
 [1.   1.   0.58 0.58]
 [1.   1.   0.58 0.58]]


In [1]:

# row_sum range
# each exp_val is between 1 and 1024
# we have N=3 values per row
# so row_sum ranges from 3 to 3072

# we store recip[x] = round(1/x * 1024)
# so that: exp_val * recip[x] >> 10 ≈ exp_val / x

print("// Reciprocal LUT: recip[x] = floor(1/x * 1024)")
print("// Usage: (exp_val * recip[row_sum]) >> 10")
print()

recip = []
for x in range(0, 3073):
    if x == 0:
        recip.append(0)  # undefined, set to 0
    else:
        val = int(round(1024.0 / x))
        recip.append(val)

# print first 20
for i in range(20):
    print(f"recip_lut[{i}] = {recip[i]};")

# check accuracy
print("\nAccuracy check:")
print(f"1/256  exact={1/256:.6f}  lut={recip[256]/1024:.6f}")
print(f"1/512  exact={1/512:.6f}  lut={recip[512]/1024:.6f}")
print(f"1/1024 exact={1/1024:.6f} lut={recip[1024]/1024:.6f}")
print(f"1/2048 exact={1/2048:.6f} lut={recip[2048]/1024:.6f}")
print(f"1/3072 exact={1/3072:.6f} lut={recip[3072]/1024:.6f}")

// Reciprocal LUT: recip[x] = floor(1/x * 1024)
// Usage: (exp_val * recip[row_sum]) >> 10

recip_lut[0] = 0;
recip_lut[1] = 1024;
recip_lut[2] = 512;
recip_lut[3] = 341;
recip_lut[4] = 256;
recip_lut[5] = 205;
recip_lut[6] = 171;
recip_lut[7] = 146;
recip_lut[8] = 128;
recip_lut[9] = 114;
recip_lut[10] = 102;
recip_lut[11] = 93;
recip_lut[12] = 85;
recip_lut[13] = 79;
recip_lut[14] = 73;
recip_lut[15] = 68;
recip_lut[16] = 64;
recip_lut[17] = 60;
recip_lut[18] = 57;
recip_lut[19] = 54;

Accuracy check:
1/256  exact=0.003906  lut=0.003906
1/512  exact=0.001953  lut=0.001953
1/1024 exact=0.000977 lut=0.000977
1/2048 exact=0.000488 lut=0.000000
1/3072 exact=0.000326 lut=0.000000


In [2]:
import numpy as np

RECIP_SCALE = 2**20  # 1048576

print(f"// Reciprocal LUT: recip[x] = floor({RECIP_SCALE}/x)")
print(f"// Usage: (exp_val * recip[row_sum]) >> 20")
print()

recip = []
for x in range(0, 3073):
    if x == 0:
        recip.append(0)
    else:
        val = int(1048576 / x)
        recip.append(val)

# print first 20
for i in range(20):
    print(f"recip_lut[{i:4d}] = {recip[i]};")

print("\n...")

# print around our expected row_sum values
# our row_sums will be around 1024+377+139 = 1540 etc
# let's check a few key values
print("\nKey values:")
for x in [341, 377, 512, 1024, 1540, 1777, 2048, 2048+377, 3072]:
    if x < len(recip):
        exact = 1.0/x
        approx = recip[x] / 1048576
        error = abs(exact - approx) / exact * 100
        print(f"1/{x:4d}: exact={exact:.8f}  lut={approx:.8f}  error={error:.4f}%")

print("\nAccuracy check:")
print(f"1/256:  exact={1/256:.6f}   lut={recip[256]/1048576:.6f}")
print(f"1/512:  exact={1/512:.6f}   lut={recip[512]/1048576:.6f}")
print(f"1/1024: exact={1/1024:.6f}  lut={recip[1024]/1048576:.6f}")
print(f"1/2048: exact={1/2048:.6f}  lut={recip[2048]/1048576:.6f}")
print(f"1/3072: exact={1/3072:.6f}  lut={recip[3072]/1048576:.6f}")

// Reciprocal LUT: recip[x] = floor(1048576/x)
// Usage: (exp_val * recip[row_sum]) >> 20

recip_lut[   0] = 0;
recip_lut[   1] = 1048576;
recip_lut[   2] = 524288;
recip_lut[   3] = 349525;
recip_lut[   4] = 262144;
recip_lut[   5] = 209715;
recip_lut[   6] = 174762;
recip_lut[   7] = 149796;
recip_lut[   8] = 131072;
recip_lut[   9] = 116508;
recip_lut[  10] = 104857;
recip_lut[  11] = 95325;
recip_lut[  12] = 87381;
recip_lut[  13] = 80659;
recip_lut[  14] = 74898;
recip_lut[  15] = 69905;
recip_lut[  16] = 65536;
recip_lut[  17] = 61680;
recip_lut[  18] = 58254;
recip_lut[  19] = 55188;

...

Key values:
1/ 341: exact=0.00293255  lut=0.00293255  error=0.0001%
1/ 377: exact=0.00265252  lut=0.00265217  error=0.0133%
1/ 512: exact=0.00195312  lut=0.00195312  error=0.0000%
1/1024: exact=0.00097656  lut=0.00097656  error=0.0000%
1/1540: exact=0.00064935  lut=0.00064850  error=0.1312%
1/1777: exact=0.00056275  lut=0.00056267  error=0.0139%
1/2048: exact=0.00048828  lut=0.00048828  error=

In [5]:
# generate full reciprocal LUT file
RECIP_SCALE = 2**20

with open('data/recip_lut.txt', 'w') as f:
    for x in range(3073):
        if x == 0:
            f.write("00000000\n")  # hex, undefined
        else:
            val = int(1048576 / x)
            f.write(f"{val:08x}\n")  # write as hex

print("recip_lut.txt written successfully")
print(f"Total entries: 3073")
print(f"Max value: {1048576} = 0x{1048576:08x}")

# also verify a few entries
print("\nVerification:")
vals = [1, 2, 3, 341, 1024, 2048, 3072]
with open('data/recip_lut.txt', 'r') as f:
    lines = f.readlines()
for v in vals:
    print(f"recip[{v}] = {int(lines[v].strip(), 16)}")

recip_lut.txt written successfully
Total entries: 3073
Max value: 1048576 = 0x00100000

Verification:
recip[1] = 1048576
recip[2] = 524288
recip[3] = 349525
recip[341] = 3075
recip[1024] = 1024
recip[2048] = 512
recip[3072] = 341


In [4]:

os.chdir(r"C:\Users\Mineth De Croos\Desktop\AI\Arch labs\attention_hardware")

# our test data
X = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 1, 0, 0]
], dtype=np.int8)

WQ = np.array([
    [ 1,  0,  1,  0],
    [ 0,  1,  0,  1],
    [ 1,  0, -1,  0],
    [ 0,  1,  0, -1]
], dtype=np.int8)

WK = np.array([
    [ 0,  1,  0,  1],
    [ 1,  0,  1,  0],
    [ 0, -1,  0,  1],
    [ 1,  0, -1,  0]
], dtype=np.int8)

WV = np.array([
    [ 1,  0,  0,  1],
    [ 0,  1,  1,  0],
    [ 1,  0,  0, -1],
    [ 0,  1, -1,  0]
], dtype=np.int8)

def save_matrix(filename, mat):
    with open(filename, 'w') as f:
        for row in mat:
            for val in row:
                # convert to python int first then mask
                f.write(f"{int(val) & 0xFF:02x}\n")

save_matrix("data/X.txt",  X)
save_matrix("data/WQ.txt", WQ)
save_matrix("data/WK.txt", WK)
save_matrix("data/WV.txt", WV)

print("Input files generated")
print("X.txt, WQ.txt, WK.txt, WV.txt saved to data/")

Input files generated
X.txt, WQ.txt, WK.txt, WV.txt saved to data/
